# Fixed-$j$ elliptic curves over number fields — reproducible experiments

This notebook accompanies the working manuscript. The **class-group/ideal algorithm** counts and constructs *every* $K$-isomorphism class of bounded intrinsic height rather than enumerating twists of an arbitrarily chosen integral model.

The exact height is

$$H(E) = (\Lambda_{K,j;\alpha,\beta}N\mathfrak b)^{12/n},\qquad n=2,4,6.$$

Use a **SageMath kernel**; place `fixed_j_sage.py` next to this notebook. Standard rational calibrations $\alpha=\beta=1$ are used in the examples.

In [ ]:
from sage.all import *
from fixed_j_sage import (
    fixed_j_data, fixed_j_count, fixed_j_representatives,
    fixed_j_minimal_curves, intrinsic_height_direct, fixed_j_leading_constant,
    fixed_j_leading_term, fixed_j_comparison_table,
)
from IPython.display import display

## 1. Rational benchmark

For $j=6912/31$ we have $a(j)=1$, and the ideal-norm parameter is $Y=X^{1/6}$. Our formula gives twice the number of squarefree positive integers $\leq Y$.

In [ ]:
j_Q = QQ(6912)/31
assert fixed_j_data(QQ, j_Q)['a'] == 1
for Y in [1,2,3,5,10,30,100]:
    print('Y=',Y,'count=',fixed_j_count(QQ,j_Q,ideal_norm_bound=Y))
print('Smallest curves:', [(r['A'],r['B']) for r in
        fixed_j_minimal_curves(QQ,j_Q)['representatives']])

## 2. Real quadratic field: units matter

For $K=\mathbb Q(\sqrt2)$ the unit quotient has size $4$, although the ordinary ideal class group is trivial.

In [ ]:
K2 = QuadraticField(2, 'r')
j2 = K2(j_Q)
print('Class number:',K2.class_number())
print('Unit group:',K2.unit_group())
print('Counts:',[fixed_j_count(K2,j2,ideal_norm_bound=y) for y in [1,2,3,5,10]])
min2 = fixed_j_minimal_curves(K2,j2)
print('Minimum height:',min2['min_height'])
print('Parameters:',[r['parameter'] for r in min2['representatives']])

## 3. Imaginary quadratic field with nontrivial class group

For $K=\mathbb Q(\sqrt{-5})$ and $j=3456/29$, $a(j)=2$, $\mathfrak N_a$ is nonprincipal, and the smallest eligible squarefree ideal has norm $2$. The minimum height is $256$, attained by **four classes**.

In [ ]:
Km5 = QuadraticField(-5, 's')
jm5 = Km5(QQ(3456)/29)
info = fixed_j_data(Km5,jm5)
print('a(j):',info['a'],'normalizing ideal:',info['Na'])
print('Class group:',Km5.class_group())
print('Counts at Y=1,2,3,5,10:',
      [fixed_j_count(Km5,jm5,ideal_norm_bound=y) for y in [1,2,3,5,10]])
minimal = fixed_j_minimal_curves(Km5,jm5)
print('Minimum height:',minimal['min_height'])
for r in minimal['representatives']:
    print('t=',r['parameter'],'  A=',r['A'],'  B=',r['B'],
          '  N(b)=',r['ideal_norm'])
print('Independent direct-height verification:')
for r in minimal['representatives']:
    assert intrinsic_height_direct(Km5,r['A'],r['B']) == r['height']
print('All four direct heights agree with the squarefree-ideal formula.')

## 4. Comparison with main terms

The horizontal variable is $Y$, the bound on $N\mathfrak b$. The corresponding height bound is $X=(\Lambda Y)^{12/n}$. This makes the comparisons numerically manageable even though curve heights grow like $Y^6$ in the generic case.

In [ ]:
from sage.all import table
fields = [('Q',QQ,j_Q),('Q(sqrt2)',K2,j2),('Q(sqrt-5)',Km5,jm5)]
for label,K,j in fields:
    result = fixed_j_comparison_table(K,j,[10,30,100,300])
    print('\n'+label, 'main coefficient:',fixed_j_leading_constant(K,j))
    display(table(rows=[[r['Y'],r['exact'],r['main_term'].n(digits=8),
                         r['relative_error'].n(digits=6)] for r in result],
                  header_row=['Y','Exact','Main term','Relative error']))

## 5. Exceptional invariants

The functions use fourth-power-free ideals for $j=1728$ and sixth-power-free ideals for $j=0$.

In [ ]:
for special_j in [0, 1728]:
    print('j=',special_j)
    for Y in [1,2,5,10]:
        print('   Y=',Y,
              'Q count=',fixed_j_count(QQ,QQ(special_j),ideal_norm_bound=Y),
              'Q(sqrt(-5)) count=',fixed_j_count(Km5,Km5(special_j),ideal_norm_bound=Y))

## 6. Verification

Run `sage -python test_fixed_j_sage.py` in the same directory to check finite counts and explicit model classification. The independent Python baseline is `independent_example_tables.py` and its CSV file; that baseline was computed separately from the Sage implementation.